### Seq2Seq Machine Translation Lab (GRU)
### Building a Seq2Seq (Encoder–Decoder) model with GRU for English → French translation.

### 1. Setup

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from collections import Counter
import re

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

/Users/kaungkhantlin/Developer/2_2025/NLP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


### 2. Load a dataset

In [3]:
raw = load_dataset("opus_books", "en-fr", split="train[:3000]")
raw = raw.train_test_split(test_size=0.2)
print(raw)

Generating train split: 100%|██████████| 127085/127085 [00:00<00:00, 2489610.72 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 2400
    })
    test: Dataset({
        features: ['id', 'translation'],
        num_rows: 600
    })
})


### 3. Define a tokenization function

In [4]:
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

### 4. Build vocabularies
##### Special tokens include
* &lt;pad&gt; - padding 
* &lt;unk&gt; - unknown
* &lt;sos&gt; - start of the sentence
* &lt;eos&gt; - end of the sentence

In [5]:
SPECIALS = ["<pad>", "<unk>", "<sos>", "<eos>"]

def build_vocab(texts, max_size=10000):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))

    vocab = {tok: i for i, tok in enumerate(SPECIALS)}
    for word, _ in counter.most_common(max_size):
        if word not in vocab:
            vocab[word] = len(vocab)
    return vocab

source_vocab = build_vocab(
    [ex["en"] for ex in raw["train"]["translation"]],
    max_size=8000
)

target_vocab = build_vocab(
    [ex["fr"] for ex in raw["train"]["translation"]],
    max_size=8000
)

SRC_PAD = source_vocab["<pad>"]
TGT_PAD = target_vocab["<pad>"]

MAX_LEN = 40

### 5. Define an encoding function

In [6]:
def encode(text, vocab):
    tokens = tokenize(text)[:MAX_LEN-2]
    ids = [vocab["<sos>"]] + [vocab.get(t, vocab["<unk>"]) for t in tokens] + [vocab["<eos>"]]
    return ids + [vocab["<pad>"]] * (MAX_LEN - len(ids))

### 6. Define a Dataset class

In [7]:
class TranslationDataset(Dataset):
    def __init__(self, data):
        self.pairs = data["translation"]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src = encode(self.pairs[idx]["en"], source_vocab)
        tgt = encode(self.pairs[idx]["fr"], target_vocab)
        return torch.tensor(src), torch.tensor(tgt)

train_ds = TranslationDataset(raw["train"])
val_ds = TranslationDataset(raw["test"])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

### 7. Encoder class

In [8]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=SRC_PAD)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        emb = self.embedding(x)
        _, hidden = self.gru(emb)
        return hidden

### 8. Decoder class

In [9]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=TGT_PAD)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        emb = self.embedding(x)
        out, hidden = self.gru(emb, hidden)
        logits = self.fc(out)
        return logits, hidden

### 9. Decoder wrapper

In [10]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        hidden = self.encoder(src)
        outputs, _ = self.decoder(tgt[:, :-1], hidden)
        return outputs

model = Seq2Seq(
    Encoder(len(source_vocab)),
    Decoder(len(target_vocab))
).to(DEVICE)

### 10. Training

In [11]:
criterion = nn.CrossEntropyLoss(ignore_index=TGT_PAD)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### 11. Training Loop

In [12]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0
    for source, target in loader:
        source, target = source.to(DEVICE), target.to(DEVICE)

        optimizer.zero_grad()
        output = model(source, target)

        loss = criterion(
            output.reshape(-1, output.size(-1)),
            target[:, 1:].reshape(-1)
        )
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)


### 12. Train the model

In [13]:
for epoch in range(5):
    loss = train_epoch(model, train_loader)
    print(f"Epoch {epoch+1}: Loss={loss:.4f}")

Epoch 1: Loss=6.9178
Epoch 2: Loss=5.8917
Epoch 3: Loss=5.3452
Epoch 4: Loss=4.9074
Epoch 5: Loss=4.5293


### 13. Inference

In [14]:
inv_target_vocab = {i: w for w, i in target_vocab.items()}

def translate(sentence):
    model.eval()
    src = torch.tensor([encode(sentence, source_vocab)]).to(DEVICE)
    hidden = model.encoder(src)

    input_tok = torch.tensor([[target_vocab["<sos>"]]]).to(DEVICE)
    result = []

    for _ in range(MAX_LEN):
        out, hidden = model.decoder(input_tok, hidden)
        next_tok = out.argmax(-1)
        token_id = next_tok.item()
        if inv_target_vocab[token_id] == "<eos>":
            break
        result.append(inv_target_vocab[token_id])
        input_tok = next_tok

    return " ".join(result)

### 14. Try Seq2Seq model out!!

In [15]:
print(translate("I like cats"))
print(translate("This class is very interesting"))

je ne sais pas
c est là
